<a href="https://colab.research.google.com/github/syk-land/gitrepo/blob/main/vlm_%ED%98%B8%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 새 섹션

In [ ]:
# ============================================================
# [Google Colab 무료 T4 GPU] 외부 호출용 이미지 생성 API 서버 (OOM 완전 해결판)
# ============================================================
!pip uninstall -y torchao
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import gc, io, re, time, threading, subprocess, traceback
import torch
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
import uvicorn
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, EulerAncestralDiscreteScheduler

# 기존 GPU 메모리 정리
for var in ["pipe", "vae"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.is_available(), "❌ GPU가 꺼져 있습니다!"
print("✅ GPU 확인:", torch.cuda.get_device_name(0))

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("nerijs/pixel-art-xl", weight_name="pixel-art-xl.safetensors")
pipe.fuse_lora(lora_scale=1.0)
pipe.enable_attention_slicing()
pipe.vae.enable_slicing()
pipe.enable_model_cpu_offload()  # T4 VRAM OOM 방지 핵심
print("✅ 모델 로딩 완료!")

app = FastAPI()
gpu_lock = threading.Lock()

class GenRequest(BaseModel):
    prompt: str
    negative_prompt: str = "3d, realistic, blurry, multiple characters, sprite sheet, grass, floor, shadow, text, watermark"
    width: int = 768
    height: int = 768
    steps: int = 20
    guidance_scale: float = 7.0
    seed: int | None = None

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0)}

@app.post("/generate")
def generate(req: GenRequest):
    try:
        with gpu_lock:
            generator = torch.Generator("cuda").manual_seed(req.seed) if req.seed is not None else None
            with torch.inference_mode():
                img = pipe(
                    prompt=req.prompt,
                    negative_prompt=req.negative_prompt,
                    width=req.width,
                    height=req.height,
                    num_inference_steps=req.steps,
                    guidance_scale=req.guidance_scale,
                    generator=generator,
                ).images[0]
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            return Response(content=buf.getvalue(), media_type="image/png")
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e), "trace": traceback.format_exc()})

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True
).start()
time.sleep(2)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not public_url:
        public_url = m.group(0)
        print("\n" + "="*60)
        print(f"🌐 외부 API URL: {public_url}")
        print(f"   상태 확인: {public_url}/health")
        print(f"   이미지 생성: POST {public_url}/generate")
        print("="*60 + "\n")
        break

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    proc.terminate()

✅ GPU 확인: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


... (14 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]